# AZ Watch Subscriber Churn & Segmentation

This project analyzes subscriber behavior for **AZ Watch**, an educational video-streaming platform. The goal is to predict subscriber churn and identify behavioral segments that can support marketing and retention strategies.

![AZ Watch marketing analytics](marketinganalytics.jpg)

## Project goals
- Predict whether a subscriber will churn using classification models.
- Compare Logistic Regression, Decision Tree, and Random Forest performance.
- Segment subscribers using K-Means clustering based on engagement behavior.
- Identify useful behavioral patterns for future subscriber personas.

## Dataset
The dataset contains 1,000 subscribers and the following variables:

| Feature | Description |
|---|---|
| `subscriber_id` | Unique subscriber identifier |
| `age_group` | Subscriber age category |
| `engagement_time` | Average minutes spent per session |
| `engagement_frequency` | Average weekly login frequency |
| `subscription_status` | Whether the subscriber remained subscribed or churned |


## 1. Load and prepare the data


In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans

df = pd.read_csv('data/AZWatch_subscribers.csv')
df.head()


In [ ]:
X = df.drop(['subscriber_id', 'subscription_status'], axis=1)
y = df['subscription_status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# One-hot encode the categorical age_group feature.
X_train_prepared = pd.get_dummies(X_train, columns=['age_group'])
X_test_prepared = pd.get_dummies(X_test, columns=['age_group']).reindex(
    columns=X_train_prepared.columns, fill_value=False
)


## 2. Churn prediction


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=10, max_depth=3, random_state=42)
}

scores = {}
for name, model in models.items():
    model.fit(X_train_prepared, y_train)
    scores[name] = model.score(X_test_prepared, y_test)
    print(f'{name} accuracy: {scores[name]:.3f}')


### Model comparison


In [ ]:
score_df = pd.DataFrame({'Model': scores.keys(), 'Accuracy': scores.values()})
score_df.sort_values('Accuracy', ascending=False)


## 3. Subscriber segmentation with K-Means


In [ ]:
segmentation = X.drop(columns=['age_group']).copy()
scaler = StandardScaler()
segmentation_normalized = scaler.fit_transform(segmentation)

sse = {}
for k in range(1, 20):
    kmeans = KMeans(n_clusters=k, random_state=1, n_init=10)
    kmeans.fit(segmentation_normalized)
    sse[k] = kmeans.inertia_

plt.figure(figsize=(8, 5))
sns.pointplot(x=list(sse.keys()), y=list(sse.values()))
plt.title('Elbow Method to Choose k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('SSE')
plt.tight_layout()
plt.show()


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=1, n_init=10)
segmentation['cluster_id'] = kmeans.fit_predict(segmentation_normalized)

cluster_analysis = segmentation.groupby('cluster_id').agg(
    engagement_time=('engagement_time', 'mean'),
    engagement_frequency=('engagement_frequency', 'mean')
).round(2)

cluster_analysis


## 4. Key findings

- Logistic Regression achieved the highest test accuracy at **92.5%**.
- Decision Tree accuracy was **92.0%**, while Random Forest accuracy was **91.5%**.
- K-Means segmentation identifies three distinct engagement profiles based on average session time and weekly login frequency.

### Cluster interpretation

| Cluster | Avg. engagement time | Avg. weekly frequency | Interpretation |
|---:|---:|---:|---|
| 0 | 9.07 min | 8.65 sessions | See engagement profile in the final chart. || 1 | 3.97 min | 5.43 sessions | See engagement profile in the final chart. || 2 | 6.81 min | 18.02 sessions | See engagement profile in the final chart. |

The segmentation can be used as a starting point for subscriber personas and targeted retention campaigns. Further work could investigate churn rates within each cluster and tune the classification models with cross-validation and hyperparameter search.
